In [1]:
import sys
sys.path.append('../')

import starry.utils.env

In [2]:
import os
import torch

from starry.utils.model_factory import loadModel
from starry.utils.config import Configuration


TRAINING_DIR = os.environ.get('TRAINING_DIR')

config = Configuration.createOrLoad(os.path.join(TRAINING_DIR, 'paraff/20241130-midiseq-paraff-se-ly+cc-d256-l8-dw0.06'))
model = loadModel(config['model'], postfix='JitEnc')

checkpoint = torch.load(config.localPath(config['best']), map_location='cpu')
model.load_state_dict(checkpoint['model'], strict=False)
model.eval()

/home/camus/work/deep-starry/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-12-06 10:23:48.488984: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-06 10:23:48.489018: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-06 10:23:48.490187: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-06 10:23:48.497535: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow b

SeqShareSEJitEnc(
  (word_emb): Embedding(324, 256, padding_idx=0)
  (word_prj): Linear(in_features=256, out_features=324, bias=False)
  (latent_prj): Linear(in_features=256, out_features=256, bias=False)
  (latent_emb): Linear(in_features=256, out_features=256, bias=True)
  (position_enc): PositionalEncoding()
  (layer_norm): LayerNorm((256,), eps=1e-06, elementwise_affine=True)
  (attention): AttentionStack(
    (layer_stack): ModuleList(
      (0-7): 8 x EncoderLayer(
        (slf_attn): MultiHeadAttention(
          (w_qs): Linear(in_features=256, out_features=256, bias=False)
          (w_ks): Linear(in_features=256, out_features=256, bias=False)
          (w_vs): Linear(in_features=256, out_features=256, bias=False)
          (fc): Linear(in_features=256, out_features=256, bias=False)
          (attention): ScaledDotProductAttention(
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (dropout): Dropout(p=0.1, inplace=False)
          (layer_norm): LayerNor

In [4]:
import yaml
import os


scores = yaml.safe_load(open(os.path.expanduser('~/data/maestro-v3.0.0/2004/midiseq.yaml'), 'r'))
scores

{'MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi': ['Sustain95 Soft127 Sustain127 Sustain127 Soft127 Sustain127 Shift1000 Shift90 Vel63 On71 Soft127 Soft127 Shift100 Off71 Sustain95 Sustain95 Sustain63 Shift90 Vel47 On55 Shift10 Vel55 On71 Sustain63 Soft95 Sustain63 Sustain63 Soft95 Sustain63 Sustain95 Sustain95 Sustain127 Sustain127 Shift180 On59 Sustain127 Soft95 Shift30 Off55 Soft95 Shift130 Off59 On62 Soft95 Shift120 Off62 Shift30 Vel79 On72 Shift10 Off71 Shift10 Vel59 On67 Shift30 Off72 Shift160 Vel63 On57 Vel71 On74 Sustain95 Shift20 Off67',
  'Sustain63 Sustain0 Shift40 Vel79 On72 Shift60 Off74 Vel51 On74 Shift10 Off72 Shift40 Vel63 On72 Shift20 Vel59 On67 Shift10 Off74 Shift100 Off67 Shift50 On66 Shift180 Off66 Shift10 Off72 Off57 Vel71 On71 Shift40 Vel35 On64 Shift30 Off71 Vel47 On72 Shift80 Off72 Shift10 Vel71 On74 Shift30 Vel51 On59 Shift50 Off64 Shift120 Vel63 On62 Shift190 On66 Shift30 Off62 Shift140 Vel59 On67 Shift20 Off66 Sustain31 

In [6]:
from tqdm import tqdm

from starry.paraff.midiseq import T2I


sum_lib = {}
for name, score in tqdm(scores.items()):
	seqs = [[T2I[token] for token in seg.strip().split(' ')] for seg in score]
	summaries = []

	with torch.no_grad():
		for ids in seqs:
			input = torch.tensor([2] + ids[:510] + [3]).long()[None]
			summaries.append(model(input))

	sum_lib[name] = torch.cat(summaries, dim=0)

sum_lib

100%|██████████| 132/132 [00:17<00:00,  7.49it/s]


{'MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_05_Track05_wav.midi': tensor([[-1.8900e+00,  1.2953e+00, -2.4425e+00,  ...,  1.1984e+00,
          -3.4630e+00,  2.9849e+00],
         [-3.0613e+00,  7.7048e-01, -1.5359e+00,  ..., -2.7355e+00,
          -1.3490e+00,  1.6525e-01],
         [-3.8062e+00,  1.8640e+00, -2.1278e+00,  ..., -1.4208e-01,
          -1.0186e+00, -2.8675e+00],
         ...,
         [-1.4742e+00, -9.8830e-01, -3.1931e-01,  ..., -3.7069e+00,
          -1.5012e-01, -2.4010e+00],
         [-2.0125e+00, -1.4042e+00, -1.5723e+00,  ..., -2.3656e+00,
           2.0257e+00, -1.5742e-01],
         [-1.2550e+00, -1.7840e+00, -5.8891e-01,  ..., -9.1249e-01,
           2.9773e+00,  1.8598e-03]]),
 'MIDI-Unprocessed_SMF_02_R1_2004_01-05_ORIG_MID--AUDIO_02_R1_2004_06_Track06_wav.midi': tensor([[-3.2060,  0.9204, -1.9228,  ..., -0.0460, -3.0615,  2.1455],
         [ 0.2007,  1.3029, -1.4917,  ..., -3.7059, -1.2918,  0.1926],
         [-3.6297, -1.0264, -1.9986,

In [10]:
next(iter(sum_lib.values())).shape

torch.Size([16, 256])

In [11]:
torch.save(sum_lib, os.path.expanduser('~/data/maestro-v3.0.0/2004/midiseq-sum.pt'))